# Corrected LoRA training on cleaned data

Follow RUN_AND_REPORT.md first. Full validation, consistent preprocessing and all-chunk encoding, epoch resume, merged-checkpoint verification and context-removal evaluation. Training is off unless explicitly enabled.

In [ ]:
import os, sys
from pathlib import Path
start = Path(os.environ.get("TOXICITY_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next(p for p in (start, *start.parents) if (p / "contextual/config.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from contextual.config import load_config, location
cfg = load_config(os.environ.get("TOXICITY_CONFIG", ROOT / "configs/contextual_clean_lora.json"))
RUN_LORA = os.environ.get("TOXICITY_RUN_LORA", "0") == "1"
print("Full training enabled:", RUN_LORA)
print("Checkpoint:", location(cfg, "checkpoint"))

In [ ]:
import torch
if RUN_LORA:
    assert torch.cuda.is_available(), "Check the GPU first; see RUN_AND_REPORT.md."
    print(torch.cuda.get_device_name(0))
else:
    print("Set TOXICITY_RUN_LORA=1 before launching Jupyter when ready to train.")

In [ ]:
if RUN_LORA:
    from contextual.adapter_finetune import run_adapter_experiment
    report = run_adapter_experiment(cfg)
    print(report["test"])
else:
    print("No training started. Re-running training resumes the last completed epoch with identical configuration.")

In [ ]:
import json
path = location(cfg, "report")
if path.exists():
    report = json.loads(path.read_text(encoding="utf-8"))
    print("Full validation:", report["validation"])
    print("With context:", report["contextual_test"])
    print("Without parents:", report["contextual_test_without_parents"])
else:
    print("Corrected full-run results do not exist yet; historical LoRA scores are not substitutes.")